In [ ]:
import os
import glob
import time
import math
import random
import itertools
from collections import defaultdict
import kagglehub
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
print(f'PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU: {torch.cuda.get_device_name(0)}')

def augment(mixture, sources):
    gain = random.uniform(0.7, 1.0)
    flip = random.random() < 0.5
    s = -1.0 if flip else 1.0
    mixture = mixture * (s * gain)
    sources = sources * (s * gain)
    if random.random() < 0.25:
        noise = torch.randn_like(mixture) * 0.003
        mixture = mixture + noise
    return (mixture, sources)

class MultiSpeakerMixtureDataset(Dataset):

    def __init__(self, data_dir):
        self.data_dir = data_dir
        manifest_path = os.path.join(data_dir, 'manifest.npy')
        if not os.path.exists(manifest_path):
            raise ValueError(f'manifest.npy not found in {data_dir}')
        manifest = np.load(manifest_path, allow_pickle=True)
        raw_speakers = {int(m['idx']): int(m['n_speakers']) for m in manifest}
        all_indices = sorted((int(os.path.basename(p).split('_')[1].split('.')[0]) for p in glob.glob(os.path.join(data_dir, 'mix_*.npy'))))
        self.indices = [idx for idx in all_indices if raw_speakers.get(idx, 0) in [2, 3, 4]]
        self.n_speakers = {idx: raw_speakers[idx] for idx in self.indices}
        print(f'Dataset Initialized: Loaded {len(self.indices)} files (Filtered strictly for 2, 3, and 4 speakers).')

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        mix_np = np.load(os.path.join(self.data_dir, f'mix_{idx:06d}.npy'), mmap_mode='r')
        src_np = np.load(os.path.join(self.data_dir, f'sources_{idx:06d}.npy'), mmap_mode='r')
        mixture = torch.from_numpy(np.array(mix_np, copy=True)).float()
        sources = torch.from_numpy(np.array(src_np, copy=True)).float()
        (mixture, sources) = augment(mixture, sources)
        return {'mixture': mixture, 'sources': sources, 'n_speakers': sources.shape[0]}

def variable_speaker_collate(batch):
    mixtures = torch.stack([b['mixture'] for b in batch], dim=0)
    sources_list = [b['sources'] for b in batch]
    n_speakers = [b['n_speakers'] for b in batch]
    return {'mixture': mixtures, 'sources_list': sources_list, 'n_speakers': n_speakers}

def make_speaker_sorted_loader(dataset, batch_size, shuffle=True, **kwargs):
    if isinstance(dataset, torch.utils.data.Subset):
        base_dataset = dataset.dataset
        subset_positions = dataset.indices
    else:
        base_dataset = dataset
        subset_positions = list(range(len(dataset)))
    groups = defaultdict(list)
    for (pos_in_subset, pos_in_base) in enumerate(subset_positions):
        file_idx = base_dataset.indices[pos_in_base]
        n = base_dataset.n_speakers[file_idx]
        groups[n].append(pos_in_subset)
    batches = []
    for n in sorted(groups.keys()):
        idxs = groups[n]
        if shuffle:
            random.shuffle(idxs)
        for start in range(0, len(idxs), batch_size):
            batch = idxs[start:start + batch_size]
            if len(batch) > 0:
                batches.append(batch)
    if shuffle:
        random.shuffle(batches)
    return DataLoader(dataset, batch_sampler=batches, collate_fn=variable_speaker_collate, **kwargs)
from scipy.io import wavfile
from IPython.display import Audio, display

In [ ]:
class ResidualConvBlock(nn.Module):

    def __init__(self, channels, kernel_size=3, dilation=1):
        super().__init__()
        pad = dilation * (kernel_size - 1) // 2
        self.net = nn.Sequential(nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation, groups=channels), nn.Conv1d(channels, channels, 1), nn.GroupNorm(8, channels), nn.PReLU())

    def forward(self, x):
        return x + self.net(x)

class CrossAttentionAttractor(nn.Module):

    def __init__(self, dim, n_heads=4, pool_len=64):
        super().__init__()
        self.pool_len = pool_len
        self.query = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.attn_resid = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.attn_mix = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.history_gru = nn.GRUCell(dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.fuse = nn.Linear(dim * 3, dim)

    def _pool(self, x):
        return F.adaptive_avg_pool1d(x, self.pool_len).permute(0, 2, 1)

    def forward(self, residual, mixture_latent, history_state):
        r = self._pool(residual)
        m = self._pool(mixture_latent)
        q = self.query.expand(r.size(0), -1, -1)
        (from_resid, _) = self.attn_resid(q, r, r)
        (from_mix, _) = self.attn_mix(q, m, m)
        combined = torch.cat([from_resid.squeeze(1), from_mix.squeeze(1), history_state], dim=-1)
        fingerprint = self.fuse(combined)
        new_history = self.history_gru(fingerprint, history_state)
        return (self.norm(fingerprint), new_history)

class AttractorGuidedSeparator(nn.Module):

    def __init__(self, encoder_dim=512, bottleneck_dim=256, n_heads=8, n_blocks=6):
        super().__init__()
        self.encoder_dim = encoder_dim
        self.bottleneck_dim = bottleneck_dim
        self.encoder_front = nn.Sequential(nn.Conv1d(1, encoder_dim, kernel_size=16, stride=8, padding=4), nn.GroupNorm(8, encoder_dim), nn.PReLU())
        self.encoder_deep = nn.Sequential(ResidualConvBlock(encoder_dim, dilation=1), ResidualConvBlock(encoder_dim, dilation=2), ResidualConvBlock(encoder_dim, dilation=4), ResidualConvBlock(encoder_dim, dilation=8))
        self.attractor = CrossAttentionAttractor(encoder_dim, n_heads=4, pool_len=64)
        self.film_gamma = nn.Linear(encoder_dim, encoder_dim)
        self.film_beta = nn.Linear(encoder_dim, encoder_dim)
        self.bottleneck_in = nn.Conv1d(encoder_dim, bottleneck_dim, 1)
        self.bottleneck_out = nn.Conv1d(bottleneck_dim, encoder_dim, 1)
        self.separator_blocks = nn.Sequential(*[nn.TransformerEncoderLayer(d_model=bottleneck_dim, nhead=n_heads, dim_feedforward=1024, dropout=0.1, batch_first=True, norm_first=True) for _ in range(n_blocks)])
        self.dilated_recurrent = nn.Sequential(nn.Conv1d(bottleneck_dim, bottleneck_dim, 3, padding=1, dilation=1, groups=bottleneck_dim), nn.Conv1d(bottleneck_dim, bottleneck_dim, 1), nn.PReLU(), nn.Conv1d(bottleneck_dim, bottleneck_dim, 3, padding=2, dilation=2, groups=bottleneck_dim), nn.Conv1d(bottleneck_dim, bottleneck_dim, 1), nn.PReLU(), nn.Conv1d(bottleneck_dim, bottleneck_dim, 3, padding=4, dilation=4, groups=bottleneck_dim), nn.Conv1d(bottleneck_dim, bottleneck_dim, 1), nn.PReLU())
        self.mask_gen = nn.Sequential(nn.Conv1d(encoder_dim, encoder_dim, 1), nn.PReLU(), nn.Conv1d(encoder_dim, encoder_dim, 1), nn.Sigmoid())
        self.decoder_proj = nn.Conv1d(encoder_dim * 2, encoder_dim, 1)
        self.decoder = nn.ConvTranspose1d(encoder_dim, 1, kernel_size=16, stride=8, padding=4)
        self.stopper = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(encoder_dim, 64), nn.PReLU(), nn.Linear(64, 1))
        self.register_buffer('pe', self._build_pe(10000, bottleneck_dim), persistent=False)

    def _build_pe(self, max_len, dim):
        pe = torch.zeros(max_len, dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)

    def encode(self, waveform):
        return self.encoder_deep(self.encoder_front(waveform))

    def extract_one(self, residual_latent, mixture_latent, original_length=None, history_state=None):
        if history_state is None:
            history_state = torch.zeros(residual_latent.size(0), self.encoder_dim, device=residual_latent.device)
        (voice_fingerprint, new_history) = self.attractor(residual_latent, mixture_latent, history_state)
        gamma = self.film_gamma(voice_fingerprint).unsqueeze(-1)
        beta = self.film_beta(voice_fingerprint).unsqueeze(-1)
        conditioned = residual_latent * gamma + beta
        x = self.bottleneck_in(conditioned).permute(0, 2, 1)
        T = x.size(1)
        x = x + self.pe[:, :T, :].to(x.device)
        x = self.separator_blocks(x)
        x = x.permute(0, 2, 1)
        x = self.dilated_recurrent(x) + x
        sep_latent = self.bottleneck_out(x)
        mask = self.mask_gen(sep_latent)
        spk_latent = residual_latent * mask
        next_residual_latent = residual_latent - spk_latent
        decode_input = torch.cat([spk_latent, mixture_latent], dim=1)
        decode_input = self.decoder_proj(decode_input)
        dominant_audio = self.decoder(decode_input).squeeze(1)
        if original_length is not None:
            L = dominant_audio.shape[-1]
            if L > original_length:
                dominant_audio = dominant_audio[..., :original_length]
            elif L < original_length:
                dominant_audio = F.pad(dominant_audio, (0, original_length - L))
        stop_logit = self.stopper(spk_latent)
        return (dominant_audio, stop_logit, next_residual_latent, new_history)

    def forward(self, waveform, max_speakers=4):
        if waveform.dim() == 2:
            waveform = waveform.unsqueeze(1)
        original_length = waveform.shape[-1]
        mixture_latent = self.encode(waveform)
        residual_latent = mixture_latent.clone()
        history_state = torch.zeros(waveform.size(0), self.encoder_dim, device=waveform.device)
        separated = []
        for _ in range(max_speakers):
            (audio, stop_logit, residual_latent, history_state) = self.extract_one(residual_latent, mixture_latent, original_length, history_state)
            separated.append(audio)
            if torch.sigmoid(stop_logit).mean().item() > 0.5:
                break
        return torch.stack(separated, dim=1)

In [ ]:
def compute_pairwise_sisdr(ests, tgts, eps=1e-08):
    ests_zm = ests - ests.mean(dim=-1, keepdim=True)
    tgts_zm = tgts - tgts.mean(dim=-1, keepdim=True)
    dot = (ests_zm * tgts_zm).sum(dim=-1)
    energy = (tgts_zm ** 2).sum(dim=-1) + eps
    proj = dot.unsqueeze(-1) / energy.unsqueeze(-1) * tgts_zm
    noise = ests_zm - proj
    ratio = (proj ** 2).sum(dim=-1) / ((noise ** 2).sum(dim=-1) + eps)
    return 10 * torch.log10(ratio + eps)

def upit_sisdr_loss(ests, tgts):
    (B, K, T) = ests.shape
    pairwise_sisdr = compute_pairwise_sisdr(ests.unsqueeze(2), tgts.unsqueeze(1))
    best_sisdr = torch.full((B,), -float('inf'), device=ests.device)
    for perm in itertools.permutations(range(K)):
        perm_tensor = torch.tensor(perm, device=ests.device)
        current_sisdr = pairwise_sisdr[:, torch.arange(K), perm_tensor].mean(dim=-1)
        best_sisdr = torch.maximum(best_sisdr, current_sisdr)
    avg_sisdr_db = best_sisdr.mean()
    return (-avg_sisdr_db, avg_sisdr_db)

def train_batch_upit(model, optimizer, mixtures, sources):
    device = mixtures.device
    model.train()
    optimizer.zero_grad(set_to_none=True)
    (B, K, T) = sources.shape
    original_length = mixtures.size(-1)
    mixture_latent = model.encode(mixtures.unsqueeze(1))
    residual_latent = mixture_latent.clone()
    history_state = torch.zeros(B, model.encoder_dim, device=device)
    extracted_audios = []
    stop_logits = []
    for step in range(K + 1):
        (audio, stop_logit, residual_latent, history_state) = model.extract_one(residual_latent, mixture_latent, original_length, history_state)
        if step < K:
            extracted_audios.append(audio)
        stop_logits.append(stop_logit.squeeze(-1))
    ests_tensor = torch.stack(extracted_audios, dim=1)
    (loss_sisdr, avg_db) = upit_sisdr_loss(ests_tensor, sources)
    stop_targets = torch.zeros((B, K + 1), device=device)
    stop_targets[:, -1] = 1.0
    stop_preds = torch.stack(stop_logits, dim=1)
    loss_stop = F.binary_cross_entropy_with_logits(stop_preds, stop_targets)
    total_loss = loss_sisdr + 0.5 * loss_stop
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
    return (total_loss.item(), avg_db.item(), loss_stop.item())

def run_training():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data_dir = '/kaggle/input/datasets/dakshgoyal387/librispeech-mixtures'
    full_dataset = MultiSpeakerMixtureDataset(data_dir)
    val_size = max(int(0.05 * len(full_dataset)), 64)
    train_size = len(full_dataset) - val_size
    (train_dataset, val_dataset) = torch.utils.data.random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
    model = AttractorGuidedSeparator(encoder_dim=512, bottleneck_dim=256, n_heads=8, n_blocks=6).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=0.0006, weight_decay=0.01)
    epochs = 120
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    best_val_sisdr = -float('inf')
    os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
    ckpt_path = '/kaggle/working/checkpoints/best_model.pt'
    start_epoch = 0
    if os.path.exists(ckpt_path):
        print(f'Resuming checkpoint from {ckpt_path}...')
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_val_sisdr = ckpt.get('val_sisdr', -float('inf'))
        print(f'Resumed from Epoch {start_epoch} | Best Val SI-SDR: {best_val_sisdr:.2f} dB')
    session_start_time = time.time()
    for epoch in range(start_epoch, epochs):
        if (time.time() - session_start_time) / 3600 > 11.25:
            print(' Approaching 12-hour Kaggle limit! Cleanly breaking loop to save final outputs...')
            break
        t0 = time.time()
        train_loader = make_speaker_sorted_loader(train_dataset, batch_size=24, shuffle=True, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=2)
        val_loader = make_speaker_sorted_loader(val_dataset, batch_size=24, shuffle=False, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=2)
        (train_loss_acc, train_db_acc) = (0.0, 0.0)
        for (batch_idx, batch) in enumerate(train_loader):
            mixtures = batch['mixture'].to(device, non_blocking=True)
            sources = torch.stack(batch['sources_list']).to(device, non_blocking=True)
            (loss, db_val, stop_val) = train_batch_upit(model, optimizer, mixtures, sources)
            train_loss_acc += loss
            train_db_acc += db_val
            if batch_idx % 25 == 0:
                print(f'  [Epoch {epoch + 1} | Batch {batch_idx:04d}/{len(train_loader):04d}] Loss: {loss:.3f} | SI-SDR: {db_val:+.2f} dB | StopLoss: {stop_val:.3f}')
        avg_train_loss = train_loss_acc / len(train_loader)
        avg_train_db = train_db_acc / len(train_loader)
        model.eval()
        val_db_acc = 0.0
        val_batches = 0
        with torch.no_grad():
            for batch in val_loader:
                mixtures = batch['mixture'].to(device, non_blocking=True)
                sources = torch.stack(batch['sources_list']).to(device, non_blocking=True)
                (B, K, T) = sources.shape
                mixture_latent = model.encode(mixtures.unsqueeze(1))
                residual_latent = mixture_latent.clone()
                history_state = torch.zeros(B, model.encoder_dim, device=device)
                extracted = []
                for _ in range(K):
                    (audio, _, residual_latent, history_state) = model.extract_one(residual_latent, mixture_latent, mixtures.size(-1), history_state)
                    extracted.append(audio)
                ests_tensor = torch.stack(extracted, dim=1)
                (_, val_db) = upit_sisdr_loss(ests_tensor, sources)
                val_db_acc += val_db.item()
                val_batches += 1
        avg_val_db = val_db_acc / max(val_batches, 1)
        elapsed = time.time() - t0
        print(f'━━━ EPOCH {epoch + 1:02d}/{epochs:02d} SUMMARY ━━━ [Time: {elapsed:.1f}s]')
        print(f'    Train Loss: {avg_train_loss:.3f} | Train SI-SDR: {avg_train_db:+.2f} dB')
        print(f'    Val SI-SDR:   {avg_val_db:+.2f} dB (Target: +14.00 dB)')
        scheduler.step(avg_val_db)
        if avg_val_db > best_val_sisdr:
            best_val_sisdr = avg_val_db
            if os.path.exists(ckpt_path):
                try:
                    os.remove(ckpt_path)
                except OSError:
                    pass
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'val_sisdr': avg_val_db}, ckpt_path)
            print(f'    NEW BEST MODEL SAVED: {best_val_sisdr:+.2f} dB SI-SDR ★')
        print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n')
if __name__ == '__main__':
    run_training()

In [ ]:
def evaluate_sisdr(ests, tgts):
    (B, K, T) = ests.shape
    pairwise_sisdr = compute_pairwise_sisdr(ests.unsqueeze(2), tgts.unsqueeze(1))
    best_sisdr = torch.full((B,), -float('inf'), device=ests.device)
    for perm in itertools.permutations(range(K)):
        perm_tensor = torch.tensor(perm, device=ests.device)
        current_sisdr = pairwise_sisdr[:, torch.arange(K), perm_tensor].mean(dim=-1)
        best_sisdr = torch.maximum(best_sisdr, current_sisdr)
    return best_sisdr.mean().item()

def load_checkpoint(ckpt_path, device):
    print(f'🔄 Loading checkpoint: {ckpt_path}')
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found at '{ckpt_path}'. Verify training completed successfully.")
    model = AttractorGuidedSeparator(encoder_dim=512, bottleneck_dim=256, n_heads=8, n_blocks=6).to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    raw_state_dict = ckpt.get('model_state_dict', ckpt)
    clean_state_dict = {k.replace('_orig_mod.', ''): v for (k, v) in raw_state_dict.items()}
    model.load_state_dict(clean_state_dict, strict=True)
    model.eval()
    epoch_val = ckpt.get('epoch', 'Unknown')
    sisdr_val = ckpt.get('val_sisdr', 'N/A')
    if isinstance(sisdr_val, float):
        sisdr_val = f'{sisdr_val:+.2f} dB'
    print(f'✅ Model Ready! (Trained Epochs: {epoch_val} | Validation SI-SDR: {sisdr_val})\n')
    return model

def separate_audio(model, mixture_path, max_speakers=2, output_dir='/kaggle/working/separated_audio', sample_rate=16000):
    device = next(model.parameters()).device
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.splitext(os.path.basename(mixture_path))[0]
    print(f'🎙️ Processing mixture: {os.path.basename(mixture_path)} ...')
    if mixture_path.endswith('.npy'):
        mix_np = np.load(mixture_path)
        mixture = torch.from_numpy(np.array(mix_np, copy=True)).float().unsqueeze(0).to(device)
    elif mixture_path.endswith('.wav'):
        (sr, mix_np) = wavfile.read(mixture_path)
        if sr != sample_rate:
            print(f'⚠️ Warning: File sample rate ({sr} Hz) != {sample_rate} Hz expected.')
        mixture = torch.from_numpy(mix_np.astype(np.float32)).unsqueeze(0).to(device)
    else:
        raise ValueError('Unsupported format. Please provide a .npy or .wav file.')
    with torch.no_grad():
        ests = model(mixture, max_speakers=max_speakers)
    ests_np = ests.squeeze(0).float().cpu().numpy()
    num_extracted = ests_np.shape[0]
    print(f'🎛️ Model extracted {num_extracted} speaker stem(s) before triggering stop token.')
    for (i, audio_data) in enumerate(ests_np):
        max_val = np.max(np.abs(audio_data)) + 1e-08
        norm_audio = (audio_data / max_val * 32767).astype(np.int16)
        out_path = os.path.join(output_dir, f'{filename}_speaker_{i + 1}.wav')
        wavfile.write(out_path, sample_rate, norm_audio)
        print(f'   └─ Speaker {i + 1} saved to: {out_path}')
        display(Audio(out_path))
    print('✨ Extraction complete!\n' + '─' * 50)

def evaluate_test_set(model, data_dir, num_samples=10):
    device = next(model.parameters()).device
    manifest_path = os.path.join(data_dir, 'manifest.npy')
    if not os.path.exists(manifest_path):
        return
    print(f'📊 Benchmarking SI-SDR across {num_samples} random dataset files (2, 3, and 4 speakers)...')
    manifest = np.load(manifest_path, allow_pickle=True)
    manifest = [m for m in manifest if int(m['n_speakers']) in [2, 3, 4]]
    entries = np.random.choice(manifest, size=min(num_samples, len(manifest)), replace=False)
    (total_sisdr, valid_count) = (0.0, 0)
    with torch.no_grad():
        for (i, entry) in enumerate(entries):
            (idx, k_true) = (int(entry['idx']), int(entry['n_speakers']))
            (mix_path, src_path) = (os.path.join(data_dir, f'mix_{idx:06d}.npy'), os.path.join(data_dir, f'sources_{idx:06d}.npy'))
            if not os.path.exists(mix_path) or not os.path.exists(src_path):
                continue
            mixture = torch.from_numpy(np.load(mix_path)).float().unsqueeze(0).to(device)
            sources = torch.from_numpy(np.load(src_path)).float().unsqueeze(0).to(device)
            ests = model(mixture, max_speakers=k_true).float()
            if ests.size(1) < k_true:
                pad = torch.zeros((1, k_true - ests.size(1), ests.size(-1)), device=device)
                ests = torch.cat([ests, pad], dim=1)
            elif ests.size(1) > k_true:
                ests = ests[:, :k_true, :]
            score = evaluate_sisdr(ests, sources)
            total_sisdr += score
            valid_count += 1
            print(f'   [Sample {i + 1:02d} | ID {idx:06d} | Ground Truth: {k_true} Speakers] SI-SDR: {score:+.2f} dB')
    if valid_count > 0:
        print(f'\n🏆 Mean SI-SDR ({valid_count} files): {total_sisdr / valid_count:+.2f} dB\n' + '─' * 50)
if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Active Inference Device: {device}\n' + '─' * 50)
    ckpt_file = '/kaggle/input/datasets/dakshgoyal387/testing-model/best_model.pt'
    if not os.path.exists(ckpt_file):
        ckpt_file = glob.glob('/kaggle/working/checkpoints/*.pt')[0] if glob.glob('/kaggle/working/checkpoints/*.pt') else ckpt_file
    model = load_checkpoint(ckpt_file, device)
    dataset_dir = '/kaggle/input/datasets/dakshgoyal387/librispeech-mixtures'
    if os.path.exists(dataset_dir):
        evaluate_test_set(model, dataset_dir, num_samples=100)
        sample_files = glob.glob(os.path.join(dataset_dir, 'mix_*.npy'))
        if sample_files:
            separate_audio(model, mixture_path=sample_files[5], max_speakers=3)